<a href="https://colab.research.google.com/github/Akshaya200722/AGENTIC_AI/blob/main/5_RAGwithLLMgeneration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers faiss-cpu groq python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 2.7 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from groq import Groq
from dotenv import load_dotenv

In [4]:
import os

os.environ["GROQ_API_KEY"] = "grok_api_key"

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

print("Groq client initialized successfully!")

Groq client initialized successfully!


In [5]:
print("Loading embedding model...")

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [7]:
import os
from google.colab import files

os.makedirs("Data", exist_ok=True)

uploaded = files.upload()

for filename in uploaded.keys():
    os.rename(filename, os.path.join("Data", filename))

print("Files uploaded successfully!")
print(os.listdir("Data"))

Saving Generative_AI.txt to Generative_AI.txt
Saving NLP.txt to NLP.txt
Saving Deep_Learning.txt to Deep_Learning.txt
Saving ML.txt to ML.txt
Files uploaded successfully!
['ML.txt', 'NLP.txt', 'Deep_Learning.txt', 'Generative_AI.txt']


In [8]:
folder = "Data"

documents = []

for filename in os.listdir(folder):

    if filename.endswith(".txt"):

        path = os.path.join(folder, filename)

        with open(path, "r", encoding="utf-8") as file:
            text = file.read()

        documents.append((filename, text))

print("Documents loaded:", len(documents))

for filename, text in documents:
    print(f"- {filename}: {len(text)} characters")

Documents loaded: 4
- ML.txt: 2310 characters
- NLP.txt: 1354 characters
- Deep_Learning.txt: 1454 characters
- Generative_AI.txt: 1601 characters


In [9]:
chunk_size = 200

chunks = []
metadata = []

for filename, text in documents:

    for i in range(0, len(text), chunk_size):

        chunk = text[i:i + chunk_size]

        chunks.append(chunk)
        metadata.append(filename)

print("Total chunks:", len(chunks))

Total chunks: 36


In [10]:
embeddings = model.encode(chunks)

embeddings = np.array(embeddings).astype("float32")

print("Embeddings created successfully!")
print("Embedding shape:", embeddings.shape)

Embeddings created successfully!
Embedding shape: (36, 384)


In [11]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("FAISS vector database ready!")
print("Number of vectors:", index.ntotal)

FAISS vector database ready!
Number of vectors: 36


In [12]:
question = input("Ask your question: ")

print("\nQuestion:", question)

Ask your question: What is the difference between machine learning and deep learning?

Question: What is the difference between machine learning and deep learning?


In [13]:
question_embedding = model.encode([question])

question_embedding = np.array(question_embedding).astype("float32")

print("Question embedding created!")

Question embedding created!


In [14]:
k = 3

distances, indices = index.search(
    question_embedding,
    k
)

print("\nRetrieved Chunks")
print("=" * 60)

context = ""

for rank, idx in enumerate(indices[0]):

    print(f"\nRank {rank + 1}")
    print("Source:", metadata[idx])
    print("Distance:", distances[0][rank])
    print("Content:", chunks[idx])

    context += chunks[idx] + "\n"


Retrieved Chunks

Rank 1
Source: Deep_Learning.txt
Distance: 0.80906236
Content: Deep Learning

Deep Learning is a subset of Machine Learning that uses artificial neural networks with multiple layers to learn complex patterns from data. These networks are inspired by the structure

Rank 2
Source: ML.txt
Distance: 0.87918824
Content: earning

Deep Learning is a subset of machine learning that uses artificial neural networks with multiple layers. Deep learning can automatically learn complex features from large datasets. It is wide

Rank 3
Source: ML.txt
Distance: 0.8883644
Content: Machine Learning

Machine Learning is a branch of Artificial Intelligence that enables computers to learn patterns from data without being explicitly programmed. It uses algorithms to analyze data and


In [15]:
prompt = f"""
You are a helpful AI tutor.

Answer the question using ONLY the information provided in the context.

If the answer is not available in the context, say:
"I don't have enough information in the provided documents."

Context:
{context}

Question:
{question}

Answer:
"""

print(prompt)


You are a helpful AI tutor.

Answer the question using ONLY the information provided in the context.

If the answer is not available in the context, say:
"I don't have enough information in the provided documents."

Context:
Deep Learning

Deep Learning is a subset of Machine Learning that uses artificial neural networks with multiple layers to learn complex patterns from data. These networks are inspired by the structure
earning

Deep Learning is a subset of machine learning that uses artificial neural networks with multiple layers. Deep learning can automatically learn complex features from large datasets. It is wide
Machine Learning

Machine Learning is a branch of Artificial Intelligence that enables computers to learn patterns from data without being explicitly programmed. It uses algorithms to analyze data and


Question:
What is the difference between machine learning and deep learning?

Answer:



In [19]:
models = client.models.list()

for model in models.data:
    print(model.id)


meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-86m
canopylabs/orpheus-arabic-saudi
groq/compound
whisper-large-v3-turbo
qwen/qwen3.6-27b
canopylabs/orpheus-v1-english
groq/compound-mini
openai/gpt-oss-safeguard-20b
openai/gpt-oss-120b
allam-2-7b
qwen/qwen3.8-27b
whisper-large-v3


In [20]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)

print("Response generated successfully!")

Response generated successfully!


In [21]:
answer = response.choices[0].message.content

print("\n")
print("=" * 60)
print("FINAL ANSWER")
print("=" * 60)

print(answer)



FINAL ANSWER
Machine Learning is a branch of Artificial Intelligence that enables computers to learn patterns from data without being explicitly programmed, using various algorithms to analyze data.  
Deep Learning is a subset of Machine Learning that specifically uses artificial neural networks with multiple layers; it can automatically learn complex features from large datasets and capture intricate patterns.
